# 05 - Avaliacao Comparativa Final + Insumos para o Docx e para os Slides

Objetivo: consolidar os resultados das Trilhas A (Notebook 03) e B (Notebook 04), compara-los com a literatura (Secao 3 do plano), responder ao **Objetivo Especifico 2** (binario Engajado vs. Desengajado) e produzir as tabelas/figuras finais que vao direto para:

- `mini-qualificacao/INF-009 2026.2 - Template Mini-qualificacao.docx` (Secao 11 do plano)
- Apresentacao de 8 slides (Secao 12 do plano)

Este notebook tambem aplica o **criterio de validacao da hipotese** definido na Secao 10 do plano (corroborada / parcialmente corroborada / nao corroborada).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import (
    f1_score, cohen_kappa_score, accuracy_score, recall_score, roc_auc_score,
    confusion_matrix, RocCurveDisplay
)
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

ROOT = Path.cwd().parent
FEATURES_DIR = ROOT / "datasets" / "DAiSEE" / "features"

## 1. Consolidar resultados das Trilhas A e B

In [ ]:
trilha_a = pd.read_csv(FEATURES_DIR / "resultados_trilha_a_classico.csv")
trilha_b = pd.read_csv(FEATURES_DIR / "resultados_trilha_b_dl.csv")

consolidado = pd.concat([trilha_a, trilha_b], ignore_index=True)
consolidado = consolidado.sort_values("macro_f1", ascending=False)
consolidado

## 2. Tabela comparativa com a literatura 2024+ (Secao 3 do plano)

Preencher `nosso_resultado_macro_f1`/`nosso_resultado_acc` com os valores do melhor modelo obtido acima.

In [ ]:
literatura = pd.DataFrame([
    {"referencia": "Abedi & Khan 2021 (ordinal, arXiv:2106.10882)", "acc": 0.674, "reporta_macro_f1_kappa": False},
    {"referencia": "Bag of States 2023 (arXiv:2301.06730)", "acc": 0.6658, "reporta_macro_f1_kappa": False},
    {"referencia": "Malekshahi et al. 2024 (arXiv:2405.04251)", "acc": 0.6857, "reporta_macro_f1_kappa": False},
    {"referencia": "VisioPhysioENet 2024 (arXiv:2409.16126)", "acc": 0.6309, "reporta_macro_f1_kappa": False},
    {"referencia": "EngageFormer 2025 (arXiv:2502.10813)", "acc": 0.639, "reporta_macro_f1_kappa": False},
    {"referencia": "ViBED-Net 2025 (arXiv:2510.18016)", "acc": 0.7343, "reporta_macro_f1_kappa": False},
])

melhor_modelo = consolidado.iloc[0]
nosso_resultado = pd.DataFrame([{
    "referencia": f"NOSSO PoC - {melhor_modelo['modelo']}",
    "acc": melhor_modelo["acc"],
    "reporta_macro_f1_kappa": True,
}])

comparativo_final = pd.concat([literatura, nosso_resultado], ignore_index=True).sort_values("acc", ascending=False)
comparativo_final

## 3. Bootstrap CI (macro-F1 e kappa) do melhor modelo vs. baseline trivial

Ver criterio de "corroboracao da hipotese" na Secao 10 do plano: IC 95% nao deve se sobrepor ao do baseline trivial.

In [ ]:
def bootstrap_metric(y_true, y_pred, metric_fn, n_boot=1000, **kwargs):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    n = len(y_true)
    scores = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        scores.append(metric_fn(y_true[idx], y_pred[idx], **kwargs))
    scores = np.array(scores)
    return np.percentile(scores, 2.5), np.percentile(scores, 97.5), scores.mean()

# NOTA: substituir `y_true_test`/`y_pred_test_melhor_modelo` pelas predicoes reais
# do melhor modelo (recarregar do Notebook 03 ou 04, conforme qual venceu acima).
# Exemplo de uso:
# ci_low, ci_high, mean = bootstrap_metric(y_true_test, y_pred_test_melhor_modelo, f1_score, average='macro')
# print(f"Macro-F1: {mean:.3f} (IC95%: [{ci_low:.3f}, {ci_high:.3f}])")

## 4. Objetivo Especifico 2 - Engajado vs. Desengajado (binario)

Reformular Engagement {0,1} -> Desengajado (0) e {2,3} -> Engajado (1). Meta: recall(Desengajado) >= 60% e AUC-ROC >= 0.70 (Secao 2 do plano).

In [ ]:
def to_binary_engagement(y):
    return (np.array(y) >= 2).astype(int)  # 1 = Engajado, 0 = Desengajado

# NOTA: reaproveitar X_test/y_test e as probabilidades (`predict_proba`) do melhor
# modelo classico (Notebook 03) ou do BiLSTM (Notebook 04, via softmax) para
# calcular a versao binaria abaixo. Esqueleto de exemplo:
#
# y_test_bin = to_binary_engagement(y_test)
# y_pred_bin = to_binary_engagement(melhor_modelo_obj.predict(X_test))
# recall_desengajado = recall_score(y_test_bin, y_pred_bin, pos_label=0)
# proba_engajado = melhor_modelo_obj.predict_proba(X_test)[:, 2:].sum(axis=1)  # soma P(classe2)+P(classe3)
# auc = roc_auc_score(y_test_bin, proba_engajado)
# print(f"Recall(Desengajado)={recall_desengajado:.3f} | AUC-ROC={auc:.3f}")
# RocCurveDisplay.from_predictions(y_test_bin, proba_engajado)
# plt.show()

## 5. Ablacoes (balanceamento e ordinalidade) - resumo

Preencher com os resultados reais das estrategias testadas nos Notebooks 03 e 04.

In [ ]:
ablacoes = consolidado[["modelo", "acc", "macro_f1", "kappa"]].copy()
ablacoes["delta_macro_f1_vs_baseline"] = ablacoes["macro_f1"] - ablacoes.loc[ablacoes["modelo"].str.contains("trivial"), "macro_f1"].values[0]
ablacoes.sort_values("macro_f1", ascending=False)

## 6. Decisao final sobre a hipotese (Secao 10 do plano)

Preencher manualmente apos rodar as celulas acima:

- Macro-F1 do melhor modelo: `____`
- Kappa do melhor modelo: `____`
- IC 95% se sobrepoe ao baseline trivial? `sim/nao`
- Recall(Desengajado) binario: `____` (meta >= 60%)
- AUC-ROC binario: `____` (meta >= 0.70)
- **Conclusao:** [ ] Corroborada  [ ] Parcialmente corroborada  [ ] Nao corroborada
- Justificativa (2-3 frases, para usar direto no Resumo/Discussao do docx e no slide 7):

## 7. Exportar tabelas/figuras finais para uso no docx e nos slides

In [ ]:
OUTPUT_DIR = ROOT / "mini-qualificacao" / "insumos_gerados"
OUTPUT_DIR.mkdir(exist_ok=True)

consolidado.to_csv(OUTPUT_DIR / "tabela_resultados_consolidados.csv", index=False)
comparativo_final.to_csv(OUTPUT_DIR / "tabela_comparativa_literatura.csv", index=False)
ablacoes.to_csv(OUTPUT_DIR / "tabela_ablacoes.csv", index=False)

print("Arquivos exportados em:", OUTPUT_DIR)

## Checklist de saida
- [ ] Tabela consolidada Trilha A vs. B gerada
- [ ] Tabela comparativa com a literatura preenchida com os resultados reais
- [ ] Bootstrap CI calculado para o melhor modelo vs. baseline trivial
- [ ] Versao binaria (Objetivo Especifico 2) avaliada (recall + AUC-ROC)
- [ ] Tabela de ablacoes preenchida
- [ ] Decisao sobre a hipotese registrada (Secao 10 do plano) com justificativa
- [ ] Tabelas exportadas para `mini-qualificacao/insumos_gerados/`

Com isso, todos os insumos necessarios para o docx (5 paginas) e para os 8 slides estao prontos (ver Secoes 11 e 12 do plano `PLANO_EXECUCAO_MINIQUALIFICACAO.md`).